# 5.6 poskyris. Duomenų apimties įtaka:
## sumažinto ir pilno LIEPA-1 + DEMAND rinkinių palyginimas

Lyginami du eksperimentai:
- **Sumažintas LIEPA-1 + DEMAND** – `full_liepa_unetdilatedmaskcnn_v2`
- **Pilnas LIEPA-1 + DEMAND** – `full_liepa_full_unetdilatedmaskcnn_v2`

## 1. Importai ir nustatymai

In [ ]:
# 1. Importai ir nustatymai
import pandas as pd
import numpy as np
import json, os, csv, warnings
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        9,
    'axes.titlesize':   10,
    'axes.labelsize':   10,
    'xtick.labelsize':  8.5,
    'ytick.labelsize':  9,
    'figure.dpi':       150,
})

CSV_PATH    = 'results/all_runs.csv'
RESULTS_DIR = 'results'
FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

EXP_MATCHED = 'full_liepa_unetdilatedmaskcnn_v2'         # sumažintas
EXP_FULL    = 'full_liepa_full_unetdilatedmaskcnn_v2'    # pilnas

LABEL_MATCHED = 'Sumazintas LIEPA-1 + DEMAND'
LABEL_FULL    = 'Pilnas LIEPA-1 + DEMAND'
COLOR_MATCHED = '#41AB5D'   # žalia
COLOR_FULL    = '#084594'   # tamsiai mėlyna

def lt(v, dec=3):
    """Skaičius su lietuvišku dešimtainiu kableliu."""
    return f'{v:.{dec}f}'.replace('.', ',')

print("Nustatymai įkelti.")
print(f"  Sumažintas: {EXP_MATCHED}")
print(f"  Pilnas:     {EXP_FULL}")


## 2. Duomenų nuskaitymas ir diagnostika

In [ ]:
# 2. Duomenų nuskaitymas ir diagnostika
df = pd.read_csv(CSV_PATH)

print("Stulpeliai:")
print(list(df.columns))
print(f"\nIš viso eilučių: {len(df)}")

fl = df[df['experiment_id'].str.contains('full_liepa', na=False)]
print(f"\n=== Visi full_liepa eksperimentai ({len(fl)}) ===")
display(fl[['experiment_id','model_name','loss_name','learning_rate',
            'n_fft','segment_seconds','num_eval_files','epochs',
            'best_epoch','best_val_loss','epochs_completed']])


## 3. Eksperimentų radimas

In [ ]:
# 3. Sumažinto ir pilno LIEPA eksperimentu radimas
def find_exp(exp_id, label):
    rows = df[df['experiment_id'] == exp_id]
    if len(rows) == 0:
        raise RuntimeError(f"ISPEJIMAS: '{exp_id}' ({label}) NERASTAS!")
    if len(rows) > 1:
        print(f"  {label}: rasta {len(rows)} eiluciu, naudojama paskutine.")
    print(f"Rastas: {label}  <-  {exp_id}")
    return rows.iloc[-1]

row_m = find_exp(EXP_MATCHED, LABEL_MATCHED)
row_f = find_exp(EXP_FULL,    LABEL_FULL)
print("\nAbiejų eksperimentų eilutės rastos.")


## 4. Reikšmių ištraukimas ir patikra

In [ ]:
# 4. Reikšmių ištraukimas ir patikra
def extract(row, exp_id, label):
    d = {
        'pesq_before':  float(row['mean_pesq_noisy']),
        'pesq_after':   float(row['mean_pesq_enhanced']),
        'delta_pesq':   float(row['mean_delta_pesq']),
        'stoi_before':  float(row['mean_stoi_noisy']),
        'stoi_after':   float(row['mean_stoi_enhanced']),
        'delta_stoi':   float(row['mean_delta_stoi']),
        'snr_before':   float(row['mean_snr_noisy_est']),
        'snr_after':    float(row['mean_snr_enhanced_est']),
        'delta_snr':    float(row['mean_delta_snr_est']),
        'n_files':      int(row['num_eval_files']),
        'n_skipped':    int(row['num_skipped_files'])
                        if not pd.isna(row['num_skipped_files']) else 0,
        'best_val':     float(row['best_val_loss']),
        'best_epoch':   int(row['best_epoch']),
        'epochs_done':  int(row['epochs_completed']),
        'max_epochs':   int(row['epochs']),
        'n_fft':        int(row['n_fft']),
        'seg_s':        float(row['segment_seconds']),
        'sr':           int(row['target_sample_rate']),
        'loss_name':    str(row['loss_name']),
        'lr':           float(row['learning_rate']),
        'n_params':     int(row['num_parameters'])
                        if not pd.isna(row['num_parameters']) else None,
        'early_stopped': bool(row['early_stopped']),
    }
    # Papildyti iš run_summary.json
    jp = os.path.join(RESULTS_DIR, exp_id, 'run_summary.json')
    if os.path.exists(jp):
        with open(jp, encoding='utf-8') as f:
            s = json.load(f)
        d['patience']   = s.get('patience', 'N/A')
        d['use_sched']  = s.get('use_lr_scheduler', False)
        d['sched_pat']  = s.get('lr_scheduler_patience', 'N/A')
        d['sched_fac']  = s.get('lr_scheduler_factor', 'N/A')
        print(f"  run_summary.json rastas: {jp}")
    else:
        print(f"  ISPEJIMAS: {jp} nerastas.")
        d['patience'] = d['use_sched'] = d['sched_pat'] = d['sched_fac'] = 'N/A'
    return d

print("=== Sumažintas LIEPA ===")
M = extract(row_m, EXP_MATCHED, LABEL_MATCHED)
print("=== Pilnas LIEPA ===")
F = extract(row_f, EXP_FULL, LABEL_FULL)

print()
for lbl, d in [(LABEL_MATCHED, M), (LABEL_FULL, F)]:
    print(f"\n  {lbl}")
    print(f"    DPESQ  = {lt(d['delta_pesq'])}")
    print(f"    DSTOI  = {lt(d['delta_stoi'], 4)}")
    print(f"    DSNR   = {lt(d['delta_snr'], 2)} dB")
    print(f"    failai = {d['n_files']} (praleista: {d['n_skipped']})")
    print(f"    best val nuost. = {lt(d['best_val'], 6)}  (epocha {d['best_epoch']})")
    print(f"    mokyta epochu = {d['epochs_done']} / {d['max_epochs']}")


## 5. Pilno LIEPA history.csv nuskaitymas

In [ ]:
# 5. Pilno LIEPA history.csv nuskaitymas
hist_path = os.path.join(RESULTS_DIR, EXP_FULL, 'history.csv')
if not os.path.exists(hist_path):
    raise FileNotFoundError(f"Nerastas: {hist_path}")

hist_f = pd.read_csv(hist_path)
print("history.csv stulpeliai:", list(hist_f.columns))
print(f"Eiluciu (epochu): {len(hist_f)}")
display(hist_f.head(5))

# Geriausia epocha
if 'improved' in hist_f.columns:
    _best_ep = int(hist_f[hist_f['improved']==True]['epoch'].max())
    print(f"\nPaskutine improved=True epocha: {_best_ep}")
else:
    _best_ep = int(hist_f.loc[hist_f['val_loss'].idxmin(), 'epoch'])
    print(f"ISPEJIMAS: 'improved' stulpelio nera – min val_loss epocha: {_best_ep}")

_final_lr = float(hist_f.iloc[-1]['learning_rate'])
print(f"Galutinis mokymosi greitis: {_final_lr}")
print(f"Epochu atlikta: {int(hist_f['epoch'].max())} / {F['max_epochs']}")


## 6. Pilno LIEPA nuostolių kreivė

In [ ]:
# 6. Loss curve generavimas – 30 pav.
_ep   = hist_f['epoch'].values
_tr   = hist_f['train_loss'].values
_vl   = hist_f['val_loss'].values
_bvl  = float(hist_f[hist_f['epoch'] == _best_ep]['val_loss'].iloc[0])
_done = int(hist_f['epoch'].max())

fig, ax = plt.subplots(figsize=(7.5, 4.0))

ax.plot(_ep, _tr, color='#2171B5', linewidth=1.4, label='Mokymo nuostolis')
ax.plot(_ep, _vl, color='#EF6C00', linewidth=1.4, linestyle='--',
        label='Validavimo nuostolis')
ax.axvline(x=_best_ep, color='#388E3C', linewidth=1.2, linestyle=':',
           label=f'Geriausia epocha ({_best_ep})')
ax.scatter([_best_ep], [_bvl], color='#388E3C', zorder=5, s=42)

ax.set_xlabel('Epocha', fontsize=10)
ax.set_ylabel('Nuostolis', fontsize=10)
ax.set_title('Pilno LIEPA-1 + DEMAND rinkinio mokymo ir\n'
             'validavimo nuostoliu kaita',
             fontsize=10, fontweight='bold', pad=8)
ax.set_xlim(0, _done + 1)
_ymax = max(float(_tr.max()), float(_vl.max())) * 1.08
ax.set_ylim(0, _ymax)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f'{v:.4f}'.replace('.', ','))
)
ax.grid(linestyle='--', linewidth=0.5, alpha=0.55)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', fontsize=8.5, framealpha=0.85)

ax.annotate(
    f'Ankstyvasis stabdymas: {_done} / {F["max_epochs"]} epochu. '
    f'Geriausias val. nuost.: {lt(_bvl, 6)} ({_best_ep} ep.)',
    xy=(0.01, 0.02), xycoords='axes fraction',
    fontsize=7.5, color='#555555', va='bottom', ha='left'
)

plt.tight_layout()
_p30 = os.path.join(FIGURES_DIR, '30_liepa_full_loss_curve.png')
_s30 = os.path.join(FIGURES_DIR, '30_liepa_full_loss_curve.svg')
fig.savefig(_p30, dpi=300, bbox_inches='tight')
fig.savefig(_s30, bbox_inches='tight')
plt.show()
print(f"Issaugota: {_p30}")
print(f"Issaugota: {_s30}")


## 7. Sumažinto ir pilno LIEPA palyginimas


In [ ]:
# 7. Palyginimo diagrama – 31 pav.
_labels = [LABEL_MATCHED, LABEL_FULL]
_colors = [COLOR_MATCHED, COLOR_FULL]
_edge   = '#333333'
_x      = np.arange(2)
_bw     = 0.50

# Trumpesni x etiketiai diagramai
_xlbls = ['Sumazintas\nLIEPA-1 + DEMAND', 'Pilnas\nLIEPA-1 + DEMAND']

fig, axes = plt.subplots(1, 3, figsize=(9.0, 3.8))

subplots = [
    (axes[0], 'delta_pesq', 'DPESQ',    3, 'DPESQ'),
    (axes[1], 'delta_stoi', 'DSTOI',    4, 'DSTOI'),
    (axes[2], 'delta_snr',  'DSNR, dB', 2, 'DSNR, dB'),
]

data_src = [M, F]

for ax, key, ylabel, dec, title_txt in subplots:
    vals = [d[key] for d in data_src]
    bars = ax.bar(_x, vals, width=_bw, color=_colors,
                  edgecolor=_edge, linewidth=0.8)

    _ymax = max(vals) * 1.22
    _ymin = 0.0
    ax.set_ylim(_ymin, _ymax)
    _pad = (_ymax - _ymin) * 0.015

    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + _pad,
            lt(v, dec),
            ha='center', va='bottom', fontsize=9, fontweight='bold'
        )

    ax.set_xticks(_x)
    ax.set_xticklabels(_xlbls, fontsize=8)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title_txt, fontsize=10, fontweight='bold', pad=6)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _, d=dec: lt(v, d))
    )
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.55)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Legenda
_patches = [
    mpatches.Patch(facecolor=COLOR_MATCHED, edgecolor=_edge, label=LABEL_MATCHED),
    mpatches.Patch(facecolor=COLOR_FULL,    edgecolor=_edge, label=LABEL_FULL),
]
fig.legend(handles=_patches, loc='lower center', bbox_to_anchor=(0.5, -0.04),
           ncol=2, fontsize=8.5, framealpha=0.9, edgecolor='#cccccc')

fig.suptitle('Sumazinto ir pilno LIEPA-1 + DEMAND rinkiniu rezultatu palyginimas',
             fontsize=10, fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0.08, 1, 1])

_p31 = os.path.join(FIGURES_DIR, '31_liepa_duomenu_apimties_palyginimas.png')
_s31 = os.path.join(FIGURES_DIR, '31_liepa_duomenu_apimties_palyginimas.svg')
fig.savefig(_p31, dpi=300, bbox_inches='tight')
fig.savefig(_s31, bbox_inches='tight')
plt.show()
print(f"Issaugota: {_p31}")
print(f"Issaugota: {_s31}")


## 8. CSV lentelių išsaugojimas

In [ ]:
# 8. CSV lentelių išsaugojimas

# --- 8a. Pilno LIEPA konfigūracija ---
csv_a = os.path.join(FIGURES_DIR, 'liepa_full_eksperimento_konfiguracija.csv')
_ankst = 'Taip' if F['early_stopped'] else 'Ne'
_npar  = f'{F["n_params"]:,}' if F['n_params'] else 'N/A'
rows_a = [
    ('Duomenu rinkinys',            'LIEPA-1 + DEMAND (pilnas)'),
    ('Modelis',                     row_f['model_name']),
    ('Nuostoliu funkcija',          F['loss_name']),
    ('Mokymosi greitis',            str(F['lr'])),
    ('STFT lango dydis',            str(F['n_fft'])),
    ('Segmento trukme',             f'{int(F["seg_s"])} s'),
    ('Diskretizavimo daznis',       f'{F["sr"]} Hz'),
    ('Optimizavimo algoritmas',     'Adam'),
    ('Modelio parametru skaicius',  _npar),
    ('Maksimalus epochu skaicius',  str(F['max_epochs'])),
    ('Faktiskai mokyta epochu',     str(F['epochs_done'])),
    ('Geriausia epocha',            str(F['best_epoch'])),
    ('Geriausias validavimo nuostolis', lt(F['best_val'], 6)),
    ('Ankstyvasis stabdymas',       _ankst),
    ('Testavimo failu skaicius',    str(F['n_files'])),
    ('Pralestu failu skaicius',     str(F['n_skipped'])),
]
with open(csv_a, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(['Parametras', 'Reiksme'])
    w.writerows(rows_a)
print(f"Issaugota: {csv_a}")

# --- 8b. Pilno LIEPA testavimo rezultatai ---
csv_b = os.path.join(FIGURES_DIR, 'liepa_full_rezultatai.csv')
with open(csv_b, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(['Rodiklis', 'Pries apdorojima', 'Po apdorojimo', 'Pokytis'])
    w.writerow(['PESQ',     lt(F['pesq_before'],3), lt(F['pesq_after'],3), lt(F['delta_pesq'],3)])
    w.writerow(['STOI',     lt(F['stoi_before'],3), lt(F['stoi_after'],3), lt(F['delta_stoi'],4)])
    w.writerow(['SNR, dB',  lt(F['snr_before'],2),  lt(F['snr_after'],2),  lt(F['delta_snr'],2)])
print(f"Issaugota: {csv_b}")

# --- 8c. Palyginimo lentelė ---
csv_c = os.path.join(FIGURES_DIR, 'liepa_duomenu_apimties_palyginimas.csv')
_hdr = ['Eksperimentas','Testavimo failu skaicius','Pralestu failu skaicius',
        'PESQ pries','PESQ po','DPESQ',
        'STOI pries','STOI po','DSTOI',
        'SNR pries, dB','SNR po, dB','DSNR, dB',
        'Geriausias validavimo nuostolis','Geriausia epocha','Faktiskai mokyta epochu']
with open(csv_c, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(_hdr)
    for lbl, d in [(LABEL_MATCHED, M), (LABEL_FULL, F)]:
        w.writerow([
            lbl,
            d['n_files'], d['n_skipped'],
            lt(d['pesq_before'],3), lt(d['pesq_after'],3), lt(d['delta_pesq'],3),
            lt(d['stoi_before'],3), lt(d['stoi_after'],3), lt(d['delta_stoi'],4),
            lt(d['snr_before'],2),  lt(d['snr_after'],2),  lt(d['delta_snr'],2),
            lt(d['best_val'],6),    d['best_epoch'],        d['epochs_done'],
        ])
print(f"Issaugota: {csv_c}")

print()
display(pd.read_csv(csv_b, encoding='utf-8-sig'))
print()
display(pd.read_csv(csv_c, encoding='utf-8-sig')[
    ['Eksperimentas','DPESQ','DSTOI','DSNR, dB','Testavimo failu skaicius']
])
